# COMP90024 Assignment 2: Data Analysis and Visualization

**Group 57**
- Josh Childs (1549150)
- Enzo Noel (1675346)
- Molly Stow (1356089)
- YU-WEI LIN (1537312)

This notebook aims to address the defined analysis goals by fetching data from the Fission backend and generating relevant visualizations.

In [ ]:
# Basic Imports and Setup
import requests
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from wordcloud import WordCloud
from datetime import datetime, timedelta

# Load functions from frontend_functions.py
# Ensure frontend_functions.py is in the same directory or Python path
try:
    from frontend_functions import (
        get_wordcloud_data, wordcloud_from_data, 
        get_top_each_platform, plot_top_each_platform,
        plot_source_sentiment, # This plots stacked area charts for overall sentiment
        plot_sentiment_across_platforms, # This plots sentiment for specific keywords/entities
        comparison_plot_keyword_counts, # For comparing keyword counts across platforms on one graph
        plot_keyword_counts, # For plotting keyword counts per platform on separate graphs
        get_entities_data, # General entity data fetcher
        labels as platform_display_labels, # Renaming to avoid conflict if 'labels' is used elsewhere
        entity_labels,
        fission as FISSION_ROUTER_URL # Use the fission URL from the imported file
    )
    print("Successfully imported from frontend_functions.py")
except ImportError as e:
    print(f"Error importing from frontend_functions.py: {e}")
    print("Please ensure frontend_functions.py is in the correct location and all its dependencies are installed.")
    # Define FISSION_ROUTER_URL if import fails, so cells below don't immediately break
    FISSION_ROUTER_URL = "http://localhost:9090" 

# Load autoreload extension for development convenience
%load_ext autoreload
%autoreload 2

# Common parameters (can be adjusted for different analyses)
DEFAULT_START_DATE = (datetime.now() - timedelta(days=30)).strftime("%Y-%m-%d") # Default to last 30 days
DEFAULT_END_DATE = datetime.now().strftime("%Y-%m-%d")
COMMON_PLATFORMS = ["reddit", "bluesky", "openaus"]

## Analysis Goal 1: What are people talking about? (topics across platforms)

**Tasks:**
- Word cloud for Bluesky, Reddit, and debate topics (OpenAustralia).
- Compare top 10 topics/entities for each platform.

In [ ]:
# --- Goal 1: Word Clouds and Top Entities ---
print(f"\n--- Running Analysis Goal 1: Topics and Entities ---")

# Fetch data for word clouds (using ORG as a proxy for general topics/organizations)
print("\nFetching data for ORG word clouds...")
org_wc_data = get_wordcloud_data("ORG") # From frontend_functions.py
if org_wc_data:
    wordcloud_from_data("ORG", org_wc_data) # From frontend_functions.py
else:
    print("Could not fetch ORG data for word clouds.")

print("\nFetching data for PERSON word clouds...")
person_wc_data = get_wordcloud_data("PER") # 'PER' might be more suitable than 'PERSON' if your NER uses PER
if person_wc_data:
    wordcloud_from_data("PER", person_wc_data) # Assuming 'PERSON' in wordcloud_from_data refers to the label for title
else:
    print("Could not fetch PERSON data for word clouds.")

print("\nFetching data for LOCATION word clouds...")
loc_wc_data = get_wordcloud_data("LOC")
if loc_wc_data:
    wordcloud_from_data("LOC", loc_wc_data)
else:
    print("Could not fetch LOCATION data for word clouds.")

# Get and plot top 10 entities for each platform of each type
entity_types_to_plot = ["ORG", "PER", "LOC"]
for entity_type in entity_types_to_plot:
    print(f"\nProcessing top entities for type: {entity_type} ({entity_labels.get(entity_type, entity_type)})")
    raw_entity_data = get_entities_data(entity_type) # Assuming this is the same as get_wordcloud_data for structure
    if raw_entity_data:
        top_entities = get_top_each_platform(raw_entity_data, count=10, normalise=False) # Get raw counts
        if top_entities:
            plot_top_each_platform(top_entities, entity_labels.get(entity_type, entity_type), normalised=False)
            
            # Create Heatmap for these top entities (using raw counts)
            try:
                # Prepare data for heatmap: entities as rows, platforms as columns, counts as values
                heatmap_df_data = {}
                all_top_entity_names = set()
                for platform_data in top_entities.values(): # platform_data is list of (entity, count) tuples
                    for entity, _ in platform_data:
                        all_top_entity_names.add(entity)
                
                for platform_name_internal, platform_data_list in top_entities.items():
                    platform_display_name = platform_display_labels.get(platform_name_internal, platform_name_internal)
                    entity_counts_for_platform = {entity: count for entity, count in platform_data_list}
                    heatmap_df_data[platform_display_name] = {entity_name: entity_counts_for_platform.get(entity_name, 0) for entity_name in all_top_entity_names}

                if heatmap_df_data and all_top_entity_names:
                    heatmap_df = pd.DataFrame(heatmap_df_data).reindex(list(all_top_entity_names))
                    heatmap_df = heatmap_df.fillna(0).astype(int) # Fill NaNs and ensure integer counts
                    # Sort by total occurrences for better visualization
                    heatmap_df['total'] = heatmap_df.sum(axis=1)
                    heatmap_df = heatmap_df.sort_values(by='total', ascending=False).drop(columns='total')
                    heatmap_df = heatmap_df.head(10) # Ensure we only plot top overall, matching the 'top_n' logic

                    if not heatmap_df.empty:
                        plt.figure(figsize=(10, 8))
                        sns.heatmap(heatmap_df, annot=True, fmt="d", cmap="viridis", linewidths=.5)
                        plt.title(f"Top Mentioned {entity_labels.get(entity_type, entity_type)} Across Platforms (Counts)", fontsize=16)
                        plt.xlabel("Platform", fontsize=12)
                        plt.ylabel(f"{entity_labels.get(entity_type, entity_type)} Entity", fontsize=12)
                        plt.xticks(rotation=45, ha='right')
                        plt.yticks(rotation=0)
                        plt.tight_layout()
                        plt.show()
                    else:
                        print(f"No data to plot heatmap for {entity_type}.")
                else:
                    print(f"Could not prepare data for heatmap for {entity_type}.")
            except Exception as e:
                print(f"Error generating heatmap for {entity_type}: {e}")
        else:
            print(f"Could not get top entities for type: {entity_type}")
    else:
        print(f"Could not fetch raw entity data for type: {entity_type}")

## Analysis Goal 2: How do the overall sentiments compare? (sentiment across platforms and time)

**Tasks:**
- Sentiment stacked bar chart across time for Bluesky, Reddit, and debates (OpenAustralia).

In [ ]:
# --- Goal 2: Overall Sentiment Comparison (Stacked Bar Charts) ---
print(f"\n--- Running Analysis Goal 2: Overall Sentiment Comparison ---")

goal2_start_date = DEFAULT_START_DATE
goal2_end_date = DEFAULT_END_DATE

print(f"Fetching overall sentiment data from {goal2_start_date} to {goal2_end_date}...")
# plot_source_sentiment from frontend_functions.py fetches data for keyword='*'
# and plots stacked AREA charts. We need to adapt it for stacked BAR charts.

# First, fetch the raw data that plot_source_sentiment would use
raw_overall_sentiment_data = None
try:
    response = requests.get(
        url=f"{FISSION_ROUTER_URL}/ui/sentiment/keyword/*/start/{goal2_start_date}/end/{goal2_end_date}",
        timeout=1500 # Increased timeout for potentially large data
    )
    response.raise_for_status()
    raw_overall_sentiment_data = response.json()
    print("Successfully fetched overall sentiment data.")
except Exception as e:
    print(f"Error fetching overall sentiment data: {e}")
    if 'response' in locals() and response is not None:
        print(f"Response text: {response.text}")

def plot_stacked_bar_sentiment(platform_name, daily_sentiment_data, start_str, end_str):
    """Plots daily sentiment as a stacked bar chart (pos, neu, neg proportions)."""
    if not daily_sentiment_data:
        print(f"No sentiment data for {platform_display_labels.get(platform_name, platform_name)} to plot.")
        return
    
    # Use dataframe function from frontend_functions.py to ensure all dates are present
    df = dataframe(daily_sentiment_data, start_str, end_str) # from frontend_functions.py
    df = df.fillna(0) # Fill NaNs for dates with no data

    if df.empty or not all(col in df.columns for col in ['neg', 'neu', 'pos', 'count']):
        print(f"Data for {platform_display_labels.get(platform_name, platform_name)} is missing required columns (neg, neu, pos, count) or is empty.")
        return

    # Calculate proportions
    # Assuming neg, neu, pos are sums, and count is the total items for that day
    df_proportions = df[['neg', 'neu', 'pos']].copy()
    # Normalize by row sum if 'count' is not directly usable or sums are already proportions
    # For this example, let's assume neg, neu, pos from backend are already sums that need to be normalized by their total for proportion
    # Or, if they are already proportions relative to 'count', this step might differ.
    # Based on plot_source_sentiment, it seems like it calculates proportions from sums.
    df_proportions['total_sentiment_sum'] = df_proportions[['neg', 'neu', 'pos']].sum(axis=1)
    for col in ['neg', 'neu', 'pos']:
        df_proportions[col] = df_proportions.apply(lambda row: row[col] / row['total_sentiment_sum'] if row['total_sentiment_sum'] > 0 else 0, axis=1)
    
    plt.figure(figsize=(15, 6))
    df_proportions[['pos', 'neu', 'neg']].plot(kind='bar', stacked=True, 
                                               color=['forestgreen', 'wheat', 'firebrick'],
                                               width=0.9)
    
    plt.title(f"Daily Sentiment Proportions on {platform_display_labels.get(platform_name, platform_name)}\n({start_str} to {end_str})")
    plt.xlabel("Date")
    plt.ylabel("Proportion of Sentiments")
    plt.xticks(rotation=45, ha='right')
    plt.legend(title='Sentiment', loc='upper left', bbox_to_anchor=(1, 1))
    plt.ylim(0, 1)
    plt.tight_layout()
    plt.show()

if raw_overall_sentiment_data:
    for platform in COMMON_PLATFORMS:
        if platform in raw_overall_sentiment_data:
            print(f"\nPlotting stacked bar sentiment for {platform_display_labels.get(platform, platform)}...")
            plot_stacked_bar_sentiment(platform, raw_overall_sentiment_data[platform], goal2_start_date, goal2_end_date)
        else:
            print(f"No overall sentiment data found for {platform}.")
else:
    print("Skipping Goal 2 plots due to data fetching error.")

## Analysis Goal 3: How do the sentiments of discussions on topics and on (or by) parties or people compare?

**Tasks:**
- Average sentiment over discussions (in debates and online) about key topics (e.g., housing).
- Average sentiment over discussions (in debates and online) about and BY key people and parties.

In [ ]:
# --- Goal 3: Sentiment of Discussions on Topics/People/Parties ---
print(f"\n--- Running Analysis Goal 3: Sentiment of Discussions ---")

key_topic = "housing"
key_people = ["Anthony Albanese", "Peter Dutton"] # Example people
key_parties = ["Labor Party", "Liberal Party"] # Example parties (ensure these match entities from NER)

print(f"\nFetching sentiment for topic: '{key_topic}'")
# plot_sentiment_across_platforms from frontend_functions.py is suitable here
# It calls /ui/sentiment-averager/type/{keyword_type} with a list of keywords
topic_sentiment_results = plot_sentiment_across_platforms(
    fission_url=FISSION_ROUTER_URL, 
    keyword_list=[key_topic],
    keyword_type="topic" # Assuming your backend /ui/sentiment-averager can handle 'topic' type
)

print(f"\nFetching sentiment for people: {key_people}")
people_sentiment_results = plot_sentiment_across_platforms(
    fission_url=FISSION_ROUTER_URL, 
    keyword_list=key_people,
    keyword_type="people" # Corresponds to 'PER' for NER, or how your backend handles it
)

print(f"\nFetching sentiment for parties: {key_parties}")
parties_sentiment_results = plot_sentiment_across_platforms(
    fission_url=FISSION_ROUTER_URL, 
    keyword_list=key_parties,
    keyword_type="parties" # Corresponds to 'ORG' for NER, or how your backend handles it
)

# Note: Sentiment OF speeches BY people/parties requires a different backend logic/endpoint
# The current plot_sentiment_across_platforms shows sentiment of discussions ABOUT these keywords.
print("\nGoal 3 Note: The plots above show sentiment of discussions ABOUT the keywords.")
print("Analyzing sentiment OF speeches BY specific people/parties would require a dedicated backend function and data source (e.g., OpenAustralia transcripts filtered by speaker).")

## Analysis Goal 4: Do people discuss the recent topics of debates?

**Tasks:**
- Choose a topic (e.g., housing).
- Graph over time Bluesky/Reddit posts with that topic AND debates with that topic.
- See if discussions rise immediately after (maybe start from a specific date of a debate with a clear topic?).

In [ ]:
# --- Goal 4: Discussion of Debate Topics ---
print(f"\n--- Running Analysis Goal 4: Discussion of Debate Topics ---")

goal4_topic = "housing"
goal4_start_date = (datetime.now() - timedelta(days=90)).strftime("%Y-%m-%d") # Look at last 90 days
goal4_end_date = DEFAULT_END_DATE

print(f"Plotting counts for topic '{goal4_topic}' from {goal4_start_date} to {goal4_end_date} across platforms.")
# comparison_plot_keyword_counts from frontend_functions.py is suitable here
counts_data = comparison_plot_keyword_counts(
    platforms=COMMON_PLATFORMS, 
    start=goal4_start_date, 
    keyword=goal4_topic,
    normalise=True # Normalize to compare trends, or False for raw counts
)

if counts_data:
    print(f"Successfully plotted counts for topic '{goal4_topic}'.")
else:
    print(f"Could not fetch or plot counts for topic '{goal4_topic}'.")

print("\nGoal 4 Note: To see if discussions rise immediately after a specific debate, you would:")
print("1. Identify a specific date of a debate on the chosen topic.")
print("2. Adjust the start/end dates for the plot to focus around that debate date.")
print("3. Observe the 'OpenAustralia' count line (representing debates) and compare with Reddit/Bluesky lines.")

--- End of Analysis Notebook ---